# 🚀 IndoBERT Fine-Tuning — Optimized for Google Colab

**Notebook ini dikhususkan untuk dijalankan di Google Colab (GPU T4/A100).**

### ✅ Fitur Optimasi:
| Fitur | Detail |
|---|---|
| **FP16 Mixed Precision** | 2× lebih cepat, hemat VRAM |
| **Gradient Accumulation** | Simulasi batch besar tanpa OOM |
| **Batch Size Lebih Besar** | 16 (efektif 64 dengan grad. accum.) |
| **Google Drive Mount** | Data & model tersimpan permanen |
| **Auto-Checkpoint Resume** | Lanjut dari checkpoint jika sesi putus |
| **Warmup + LR Scheduler** | Cosine schedule untuk konvergensi lebih baik |

### 📋 Panduan Penggunaan:
1. **Runtime → Change runtime type → GPU (T4 atau lebih tinggi)**
2. Jalankan **Cell 1** untuk mount Google Drive
3. Pastikan data CSV sudah ada di Drive (sesuai path di Cell 2)
4. Jalankan semua cell secara berurutan

> ⚠️ Jika sesi Colab putus, jalankan ulang dari **Cell 1-4**, lalu langsung ke **Cell 7**


## ⚙️ Cell 1 — Setup: Mount Google Drive & Install Library

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Cek GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU Info:')
print(result.stdout.strip() if result.returncode == 0 else 'GPU tidak ditemukan!')

# Install library yang dibutuhkan
!pip install -q transformers==4.44.2 datasets accelerate scikit-learn seaborn joblib
print('Library siap!')


## 📁 Cell 2 — Konfigurasi Path

> **Sesuaikan `DRIVE_ROOT`** dengan folder project Anda di Google Drive.
> Struktur folder yang diharapkan:
> ```
> MyDrive/
> └── BisnisComNewsClassification/
>     ├── data/
>     │   ├── train.csv
>     │   ├── val.csv
>     │   └── test.csv
>     ├── models/
>     │   └── label_encoder.pkl
>     └── outputs/
> ```


In [ ]:
import os

# ===== KONFIGURASI — Sesuaikan path ini =====
DRIVE_ROOT = '/content/drive/MyDrive/BisnisComNewsClassification'

# Path derivatif (tidak perlu diubah)
DATA_DIR        = os.path.join(DRIVE_ROOT, 'data')
MODELS_DIR      = os.path.join(DRIVE_ROOT, 'models')
OUTPUTS_DIR     = os.path.join(DRIVE_ROOT, 'outputs')
FIGURES_DIR     = os.path.join(OUTPUTS_DIR, 'figures')
CHECKPOINT_DIR  = os.path.join(MODELS_DIR, 'indobert_checkpoints')
FINAL_MODEL_DIR = os.path.join(MODELS_DIR, 'indobert_final')

for d in [OUTPUTS_DIR, FIGURES_DIR, CHECKPOINT_DIR, FINAL_MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

# ===== Hyperparameter Training =====
MODEL_NAME              = 'indobenchmark/indobert-base-p1'
MAX_LEN                 = 256
BATCH_SIZE              = 16       # per device (GPU)
GRAD_ACCUM_STEPS        = 4        # effective batch = 16 x 4 = 64
LEARNING_RATE           = 2e-5
NUM_EPOCHS              = 5
WARMUP_RATIO            = 0.1      # 10% langkah pertama = warmup
WEIGHT_DECAY            = 0.05
EARLY_STOPPING_PATIENCE = 2

eff_batch = BATCH_SIZE * GRAD_ACCUM_STEPS
print(f'Konfigurasi berhasil:')
print(f'  Data       : {DATA_DIR}')
print(f'  Checkpoint : {CHECKPOINT_DIR}')
print(f'  Model Final: {FINAL_MODEL_DIR}')
print(f'  Effective Batch Size: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {eff_batch}')


## 📦 Cell 3 — Import Library & Cek CUDA

In [ ]:
import pandas as pd
import numpy as np
import joblib
import torch
import torch.nn as nn
import json
import time
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score,
    confusion_matrix, precision_recall_fscore_support
)
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU Aktif  : {gpu_name}')
    print(f'VRAM Total : {gpu_mem:.1f} GB')
else:
    print('PERINGATAN: GPU tidak aktif!')
    print('Aktifkan via: Runtime -> Change runtime type -> GPU')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Semua library berhasil diimport!')


## 📊 Cell 4 — Load Data

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv')).fillna('')
val_df   = pd.read_csv(os.path.join(DATA_DIR, 'val.csv')).fillna('')
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv')).fillna('')

X_train, y_train = train_df['text'].tolist(), train_df['label'].tolist()
X_val,   y_val   = val_df['text'].tolist(),   val_df['label'].tolist()
X_test,  y_test  = test_df['text'].tolist(),  test_df['label'].tolist()

le          = joblib.load(os.path.join(MODELS_DIR, 'label_encoder.pkl'))
num_classes = len(le.classes_)

n_train = len(X_train)
n_val   = len(X_val)
n_test  = len(X_test)
print(f'Statistik Data:')
print(f'  Train  : {n_train} sampel')
print(f'  Val    : {n_val} sampel')
print(f'  Test   : {n_test} sampel')
print(f'  Jumlah Kelas: {num_classes}')
print(f'  Nama Kelas  : {list(le.classes_)}')

label_counts = Counter(y_train)
print('\nDistribusi label (train):')
for cls_id, count in sorted(label_counts.items()):
    cls_name = le.classes_[cls_id] if cls_id < len(le.classes_) else f'class_{cls_id}'
    print(f'  [{cls_id}] {cls_name}: {count}')


## 🤖 Cell 5 — Load Model & Tokenizer

In [ ]:
print(f'Memuat model: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    ignore_mismatched_sizes=True
)
model = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model berhasil dimuat!')
print(f'  Total parameter : {total_params}')
print(f'  Trainable params: {trainable_params}')


## 🏗️ Cell 6 — Dataset PyTorch & Class Weights

In [ ]:
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(X_train, y_train, tokenizer)
val_dataset   = NewsDataset(X_val,   y_val,   tokenizer)
test_dataset  = NewsDataset(X_test,  y_test,  tokenizer)

n_tr = len(train_dataset)
n_vl = len(val_dataset)
n_ts = len(test_dataset)
print(f'Dataset siap: train={n_tr}, val={n_vl}, test={n_ts}')

# Class weights untuk dataset imbalanced
classes              = np.unique(y_train)
class_weights        = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print('\nClass Weights (untuk imbalanced data):')
for cls_id, w in zip(classes, class_weights):
    cls_name = le.classes_[cls_id] if cls_id < len(le.classes_) else f'class_{cls_id}'
    print(f'  [{cls_id}] {cls_name}: {w:.4f}')


## 🔁 Cell 7 — Custom Trainer & Training Arguments

> **Jika sesi Colab putus**, jalankan Cell 1-6 lalu cell ini.
> Trainer akan otomatis melanjutkan dari checkpoint terakhir di Google Drive.


In [ ]:
# Custom Trainer dengan Weighted CrossEntropy Loss
class WeightedTrainer(Trainer):
    """Trainer dengan class weights untuk menangani imbalanced dataset."""
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights)
        loss    = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted', zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

# Cek checkpoint yang ada untuk auto-resume
ckpt_dirs = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith('checkpoint')]
ckpt_dirs = sorted(ckpt_dirs,
    key=lambda x: int(x.split('-')[-1]) if x.split('-')[-1].isdigit() else 0)
resume_from = None
if ckpt_dirs:
    resume_from = os.path.join(CHECKPOINT_DIR, ckpt_dirs[-1])
    print(f'Checkpoint ditemukan! Melanjutkan dari: {resume_from}')
else:
    print('Tidak ada checkpoint. Memulai training dari awal.')

# Training Arguments — Optimized for Colab GPU
training_args = TrainingArguments(
    output_dir                  = CHECKPOINT_DIR,
    # --- Evaluasi & Checkpoint ---
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',   # Gunakan F1 untuk dataset imbalanced
    greater_is_better           = True,
    save_total_limit            = 2,      # Hemat storage Drive
    # --- Hyperparameter ---
    learning_rate               = LEARNING_RATE,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    num_train_epochs            = NUM_EPOCHS,
    weight_decay                = WEIGHT_DECAY,
    warmup_ratio                = WARMUP_RATIO,
    lr_scheduler_type           = 'cosine',  # Cosine decay setelah warmup
    # --- Optimasi Kecepatan Colab GPU ---
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    fp16                        = (DEVICE == 'cuda'),  # FP16 hanya jika GPU
    dataloader_num_workers      = 2,
    dataloader_pin_memory       = True,
    # --- Logging ---
    logging_strategy            = 'epoch',
    logging_dir                 = os.path.join(OUTPUTS_DIR, 'logs'),
    report_to                   = 'none',
    seed                        = 42,
)

trainer = WeightedTrainer(
    class_weights    = class_weights_tensor,
    model            = model,
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = val_dataset,
    processing_class = tokenizer,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)]
)

eff = BATCH_SIZE * GRAD_ACCUM_STEPS
fp16_on = (DEVICE == 'cuda')
print(f'Trainer siap!')
print(f'  Effective Batch Size : {eff}')
print(f'  FP16 Mixed Precision : {fp16_on}')
print(f'  LR Scheduler         : Cosine dengan warmup {int(WARMUP_RATIO*100)}%')
print(f'  Early Stopping       : patience={EARLY_STOPPING_PATIENCE}')


## 🏋️ Cell 8 — Mulai Training

> **Estimasi durasi di GPU T4:** ~20-40 menit per epoch (tergantung ukuran dataset)
> Checkpoint disimpan otomatis ke Google Drive setiap epoch.


In [ ]:
print('Memulai Training IndoBERT...')
print(f'  Checkpoint disimpan ke: {CHECKPOINT_DIR}')
print('-' * 60)

start_time   = time.time()
train_result = trainer.train(resume_from_checkpoint=resume_from)
elapsed      = time.time() - start_time

elapsed_min  = elapsed / 60
elapsed_hr   = elapsed / 3600
print('-' * 60)
print(f'Training selesai dalam {elapsed_min:.1f} menit ({elapsed_hr:.2f} jam)')
print(f'Total steps  : {train_result.global_step}')
print(f'Training loss: {train_result.training_loss:.4f}')


## 💾 Cell 9 — Simpan Model Final ke Google Drive

In [ ]:
print(f'Menyimpan model final ke: {FINAL_MODEL_DIR}')
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

metrics_path = os.path.join(OUTPUTS_DIR, 'indobert_train_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(train_result.metrics, f, indent=2)

print(f'Model berhasil disimpan!')
print(f'  Model     : {FINAL_MODEL_DIR}')
print(f'  Tokenizer : {FINAL_MODEL_DIR}')
print(f'  Metrik    : {metrics_path}')


## 📈 Cell 10 — Visualisasi Training History

In [ ]:
def plot_bert_training_history(trainer, save_dir=FIGURES_DIR):
    """Plot training & eval loss/accuracy/F1 dari log Trainer HuggingFace."""
    logs = trainer.state.log_history
    train_logs = [l for l in logs if 'loss' in l and 'eval_loss' not in l]
    eval_logs  = [l for l in logs if 'eval_loss' in l]

    if not eval_logs:
        print('Tidak ada log evaluasi. Pastikan training sudah selesai.')
        return

    train_epochs = [l.get('epoch', i+1) for i, l in enumerate(train_logs)]
    train_loss   = [l['loss'] for l in train_logs]
    eval_epochs  = [l['epoch'] for l in eval_logs]
    eval_loss    = [l['eval_loss'] for l in eval_logs]
    eval_acc     = [l.get('eval_accuracy', None) for l in eval_logs]
    eval_f1      = [l.get('eval_f1', None) for l in eval_logs]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('IndoBERT — Training History (Colab)', fontsize=14, fontweight='bold')

    # Plot Loss
    axes[0].plot(train_epochs, train_loss, 'o-',  color='#1f77b4', label='Train Loss', lw=2)
    axes[0].plot(eval_epochs,  eval_loss,  's--', color='#ff7f0e', label='Val Loss',   lw=2)
    best_idx = int(np.argmin(eval_loss))
    best_ep  = eval_epochs[best_idx]
    axes[0].axvline(best_ep, color='green', linestyle=':', lw=1.5, label=f'Best Ep {best_ep:.0f}')
    axes[0].set_title('Training vs Validation Loss', fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(fontsize=9)
    axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    # Plot Accuracy
    clean_acc = [v if v is not None else float('nan') for v in eval_acc]
    axes[1].plot(eval_epochs, clean_acc, 's-', color='#2ca02c', label='Val Accuracy', lw=2)
    axes[1].set_title('Validation Accuracy', fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_ylim(0, 1.05); axes[1].legend(fontsize=9)
    axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    # Plot F1
    clean_f1 = [v if v is not None else float('nan') for v in eval_f1]
    axes[2].plot(eval_epochs, clean_f1, 'd-', color='#d62728', label='Val F1 (weighted)', lw=2)
    axes[2].set_title('Validation F1-Score', fontweight='bold')
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('F1-Score')
    axes[2].set_ylim(0, 1.05); axes[2].legend(fontsize=9)
    axes[2].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.tight_layout()
    save_path = os.path.join(save_dir, 'indobert_training_history.png')
    plt.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Gambar disimpan: {save_path}')

plot_bert_training_history(trainer)


## 🧪 Cell 11 — Evaluasi pada Test Set

In [ ]:
print('Evaluasi pada Test Set...')
predictions = trainer.predict(test_dataset)
y_pred      = np.argmax(predictions.predictions, axis=1)

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print('=' * 60)
print('  HASIL EVALUASI TEST SET')
print('=' * 60)
print(f'  Accuracy : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  F1-Score : {f1:.4f}  ({f1*100:.2f}%)')
print('=' * 60)
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

results_df   = pd.DataFrame([{'Model': 'IndoBERT-Colab', 'Accuracy': acc, 'F1-Score': f1}])
results_path = os.path.join(OUTPUTS_DIR, 'indobert_results.csv')
results_df.to_csv(results_path, index=False)
print(f'Hasil disimpan ke: {results_path}')


## 📊 Cell 12 — Visualisasi Confusion Matrix

In [ ]:
def plot_confusion_matrix(y_true, y_pred, model_name, target_names, save_dir=FIGURES_DIR):
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    title = f'Confusion Matrix -- {model_name}'
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.01)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names,
                linewidths=0.5, ax=axes[0], cbar_kws={'shrink': 0.8})
    axes[0].set_title('Jumlah Prediksi (Count)', fontsize=11)
    axes[0].set_xlabel('Prediksi'); axes[0].set_ylabel('Aktual')
    axes[0].tick_params(axis='x', rotation=45)

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
                xticklabels=target_names, yticklabels=target_names,
                linewidths=0.5, vmin=0, vmax=1,
                ax=axes[1], cbar_kws={'shrink': 0.8})
    axes[1].set_title('Normalized per Kelas Aktual', fontsize=11)
    axes[1].set_xlabel('Prediksi'); axes[1].set_ylabel('Aktual')
    axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    safe_name = model_name.lower().replace(' ', '_')
    fname = os.path.join(save_dir, f'cm_{safe_name}.png')
    plt.savefig(fname, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Gambar disimpan: {fname}')

plot_confusion_matrix(y_test, y_pred, 'IndoBERT', le.classes_)


## 📊 Cell 13 — Metrik per Kelas & Ringkasan Final

In [ ]:
prec, rec, f1_per, _ = precision_recall_fscore_support(
    y_test, y_pred, labels=np.arange(len(le.classes_)), zero_division=0
)

fig, axes = plt.subplots(1, 2, figsize=(20, 5))

# Per kelas
x     = np.arange(len(le.classes_))
width = 0.25
color = '#9467bd'
bars_p = axes[0].bar(x - width, prec,   width, label='Precision', color=color, alpha=0.85)
bars_r = axes[0].bar(x,         rec,    width, label='Recall',    color=color, alpha=0.55)
bars_f = axes[0].bar(x + width, f1_per, width, label='F1-Score',  color=color, alpha=0.30)
for bar, val in zip(bars_f, f1_per):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')
axes[0].set_title('Precision / Recall / F1 per Kelas', fontsize=12, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(le.classes_, rotation=35, ha='right', fontsize=9)
axes[0].set_ylim(0, 1.18); axes[0].set_ylabel('Score')
axes[0].legend(fontsize=9)
axes[0].axhline(0.9, color='red', linestyle='--', lw=0.8, alpha=0.6)

# Ringkasan
prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
    y_test, y_pred, average='weighted', zero_division=0
)
metrics_labels = ['Accuracy', 'Precision\n(weighted)', 'Recall\n(weighted)', 'F1-Score\n(weighted)']
scores  = [acc, prec_w, rec_w, f1_w]
palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
bars = axes[1].bar(metrics_labels, scores, color=palette, alpha=0.85, width=0.5)
for bar, val in zip(bars, scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_ylim(0, 1.15); axes[1].set_ylabel('Score', fontsize=11)
axes[1].set_title('Ringkasan Metrik Test Set', fontsize=12, fontweight='bold')
axes[1].axhline(0.9, color='grey', linestyle='--', lw=0.8, alpha=0.7)

plt.suptitle('IndoBERT — Hasil Evaluasi Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(FIGURES_DIR, 'indobert_final_metrics.png')
plt.savefig(save_path, bbox_inches='tight', dpi=150)
plt.show()

print(f'Gambar disimpan: {save_path}')
print('\nRingkasan Hasil:')
print(f'  Accuracy  : {acc:.4f}')
print(f'  Precision : {prec_w:.4f}')
print(f'  Recall    : {rec_w:.4f}')
print(f'  F1-Score  : {f1_w:.4f}')


---
## ✅ Selesai!

Semua output telah disimpan ke Google Drive:

| File | Lokasi |
|------|--------|
| Model final | `models/indobert_final/` |
| Checkpoint | `models/indobert_checkpoints/` |
| Hasil CSV | `outputs/indobert_results.csv` |
| Training metrics | `outputs/indobert_train_metrics.json` |
| Grafik Training | `outputs/figures/indobert_training_history.png` |
| Confusion Matrix | `outputs/figures/cm_indobert.png` |
| Metrik Final | `outputs/figures/indobert_final_metrics.png` |
